# 04 — Preprocessing: Lending Club

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os
from pathlib import Path

_RAW_DIR = Path('../data/raw/lending_club/archive')
_CANDIDATES = [
    _RAW_DIR / 'accepted_2007_to_2018q4.csv' / 'accepted_2007_to_2018Q4.csv',
    _RAW_DIR / 'accepted_2007_to_2018Q4.csv',
    _RAW_DIR / 'accepted_2007_to_2018Q4.csv.gz',
]
RAW_PATH = next((c for c in _CANDIDATES if c.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(
        'Lending Club source not found. Expected one of:\n  '
        + '\n  '.join(str(c) for c in _CANDIDATES)
    )
print(f'Source: {RAW_PATH}')

OUT_PATH = '../data/processed/v4/lending_club_full.csv'

os.makedirs('../data/processed/v4', exist_ok=True)

## 1. Load and Filter loan_status

In [ ]:
print('Loading CSV (this takes ~1-2 min for 2.26M rows)...')
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Full shape: {df.shape}')
print(f'\nloan_status distribution:')
print(df['loan_status'].value_counts())

target_map = {'Charged Off': 1, 'Default': 1, 'Fully Paid': 0}
df = df[df['loan_status'].isin(target_map)].copy()
df['target'] = df['loan_status'].map(target_map)
df = df.drop(columns=['loan_status'])

print(f'\nAfter filter (Fully Paid / Charged Off / Default): {df.shape}')
print(f'Default rate: {df["target"].mean():.4f}  ({df["target"].sum()} defaults of {len(df)})')

## 2. Drop Leaky, ID and Text Columns

In [ ]:
leaky_cols = [
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
]

drop_cols = [
    'id', 'member_id', 'url', 'desc', 'title', 'emp_title',
    'zip_code',
    'pymnt_plan',
    'policy_code',
    'funded_amnt_inv',
]

joint_cols = [
    'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    'revol_bal_joint',
    'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc',
    'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog',
]

all_drop = set(leaky_cols + drop_cols + joint_cols)
all_drop = [c for c in all_drop if c in df.columns]
print(f'Dropping {len(all_drop)} columns (leaky + IDs + joint)')
df = df.drop(columns=all_drop)
print(f'Shape after drop: {df.shape}')

## 3. Missingness Profile (Reported Only)

In [ ]:
from src.datasets import MISSING_THRESHOLD

missing_frac = df.isnull().mean()
above = sorted(missing_frac[missing_frac > MISSING_THRESHOLD].index.tolist())
print(f'Columns with >{MISSING_THRESHOLD:.0%} missing (kept in the export; the runtime '
      f'filter decides per fit subset): {len(above)}')
for c in above:
    print(f'  {c}: {missing_frac[c]:.1%}')
print(f'\nShape unchanged: {df.shape}')

## 4. Parse String Columns

In [ ]:
if 'term' in df.columns:
    df['term'] = df['term'].str.extract(r'(\d+)').astype(float)
    print(f'term unique: {sorted(df["term"].dropna().unique().tolist())}')

if 'emp_length' in df.columns:
    emp_map = {
        '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
        '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
        '8 years': 8, '9 years': 9, '10+ years': 10, 'n/a': np.nan
    }
    df['emp_length'] = df['emp_length'].map(emp_map)
    print(f'emp_length distribution: {df["emp_length"].value_counts(dropna=False).sort_index().to_dict()}')

## 5. Drop Endogenous Pricing Columns

In [ ]:
from src.datasets import LENDING_CLUB_ENDOGENOUS_DROPS

to_drop = [c for c in LENDING_CLUB_ENDOGENOUS_DROPS if c in df.columns]
df = df.drop(columns=to_drop)
print(f'Dropped endogenous pricing columns: {to_drop}')
print(f'Kept as auxiliary (never predictors): int_rate, issue_year')
print(f'Shape: {df.shape}')

## 6. Date Columns to Credit History Length

In [ ]:
if 'issue_d' in df.columns and 'earliest_cr_line' in df.columns:
    issue_dt = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
    earliest_dt = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
    df['credit_history_months'] = (
        (issue_dt.dt.year - earliest_dt.dt.year) * 12
        + (issue_dt.dt.month - earliest_dt.dt.month)
    ).clip(lower=0, upper=720)
    df['issue_year'] = issue_dt.dt.year
    print(f'credit_history_months: mean={df["credit_history_months"].mean():.1f}, '
          f'min={df["credit_history_months"].min()}, max={df["credit_history_months"].max()}')
elif 'earliest_cr_line' in df.columns:
    earliest_dt = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
    ref_date = pd.Timestamp('2018-12-01')
    df['credit_history_months'] = (
        (ref_date.year - earliest_dt.dt.year) * 12
        + (ref_date.month - earliest_dt.dt.month)
    ).clip(lower=0, upper=720)

date_cols_to_drop = [c for c in ['issue_d', 'earliest_cr_line'] if c in df.columns]
df = df.drop(columns=date_cols_to_drop)
print(f'Dropped date strings: {date_cols_to_drop}')
print(f'Shape: {df.shape}')

## 7. Categorical Columns — Kept Raw

In [ ]:
from src.datasets import get

config = get('lending_club')
declared = list(config.categorical_cols)
present = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]

assert sorted(present) == sorted(declared), (
    f'Categorical columns disagree with src/datasets.py:\n'
    f'  in frame:  {sorted(present)}\n'
    f'  declared:  {sorted(declared)}'
)
print(f'{len(declared)} categorical columns kept raw:')
for c in declared:
    print(f'  {c}: {df[c].nunique(dropna=True)} levels, {df[c].isnull().mean():.1%} missing')
print(f'Shape unchanged: {df.shape}')

## 8. Outlier Profile (No Clipping Here)

In [ ]:
for col in config.winsor_cols:
    reference = df[col].quantile(0.99)
    print(f'{col}: max={df[col].max():,.1f}, P99 (reference only)={reference:,.1f}, '
          f'above: {(df[col] > reference).sum()} values')

## 9. Missing Values (Reported Only)

In [ ]:
missing_before = df.isnull().sum()
missing_cols = missing_before[missing_before > 0]
print(f'Columns with missing values (kept as NaN, imputed per fit subset): {len(missing_cols)}')
print(missing_cols.sort_values(ascending=False).head(20))

print(f'\nTotal missings (expected > 0): {df.isnull().sum().sum()}')

## 10. No Subsample

In [ ]:
print(f'Full preprocessed shape: {df.shape}')
print(f'Default rate: {df["target"].mean():.4f}')
print(f'issue_year range: {int(df["issue_year"].min())}-{int(df["issue_year"].max())}')
print(f'mean int_rate (full frame, reference only): {df["int_rate"].mean():.2f}%')
print('  ROI is computed from the training partition at runtime, not from this mean.')

## 11. Sanity Checks

In [ ]:
assert df['target'].isnull().sum() == 0, 'Target has missing values!'
n_missing = df.drop(columns=['target']).isnull().sum().sum()
print(f'✓ Target complete — feature missings (pipeline imputes): {n_missing}')
assert n_missing > 0, 'Expected missing values to survive the export'

non_num = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
assert sorted(non_num) == sorted(config.categorical_cols), \
    f'Unexpected non-numeric columns: {sorted(set(non_num) ^ set(config.categorical_cols))}'
print(f'✓ {len(non_num)} categorical columns raw, all others numeric')

assert set(df['target'].unique()).issubset({0, 1}), 'Target not binary!'
print(f'✓ Target binary — Default rate: {df["target"].mean():.4f}')

n_neg = (df['target'] == 0).sum()
n_pos = (df['target'] == 1).sum()
print(f'  Classes: {n_neg} (no default) vs {n_pos} (default) → Ratio {n_neg / n_pos:.1f}:1')
print('  Imbalance via scale_pos_weight per fit call')

print(f'\nFinal shape: {df.shape}')

In [ ]:
leaky_check = ['out_prncp', 'total_pymnt', 'recoveries', 'last_pymnt_amnt',
               'total_rec_prncp', 'debt_settlement_flag']
found_leaky = [c for c in leaky_check + list(LENDING_CLUB_ENDOGENOUS_DROPS) if c in df.columns]
assert len(found_leaky) == 0, f'Leaky or endogenous columns still present: {found_leaky}'
print('✓ No leaky or endogenous pricing columns')

key_cols = [c for c in ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
                        'fico_range_low', 'term'] if c in df.columns]
df[key_cols].describe().round(2)

## 12. Save

In [ ]:
df = df.copy()
df.insert(0, 'source_row_id', np.arange(len(df), dtype=np.int64))
cols = [c for c in df.columns if c != 'target'] + ['target']
df = df[cols]

df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
from src.semiraw import audit_semiraw_export

report = audit_semiraw_export(OUT_PATH, 'lending_club', expect_missing=True)